In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
# Import the processing module from the same folder
sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_KPI_adequacy, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = False

dim = (1000,500)
g_BLUE = "1616A7"
g_GREY = "#7F7F7F"

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


In [ ]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v28.0s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v28.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v30.0.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v30.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v30.0.2s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v30.1.1s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v31.0.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v31.1.1s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)
gcd_KPI_adequacy['E_ENS_%'] = gcd_KPI_adequacy['E_ENS_MWh']/gcd_KPI_adequacy['E_input_load_MWh']*100


Data construction and filtering

In [ ]:
filter_ = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values='solution_id',
    # var_name='cost_type',
    # value_name='cost_value'
).dropna().index
gcd_KPI_adequacy = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(filter_)]
gcdi_KPI_adequacy = gcdi_KPI_adequacy[gcdi_KPI_adequacy['day'].isin(filter_)]

In [ ]:
dispatch_data = gcd_KPI_adequacy[['day','model_type', 'storage_charge_uc_MWh', 'storage_discharge_uc_MWh','thermal_production_uc_MWh']].copy()
dispatch_data['storage_charge_discharge_MWh'] = dispatch_data['storage_charge_uc_MWh'] + dispatch_data['storage_discharge_uc_MWh']

In [ ]:
def is_approx(x, y, rel_tol=1e-4):
    return abs(x - y) <= rel_tol * max(abs(x), abs(y))

def is_bigger_than(x, y, rel_tol=1e-4):
    return x > y + rel_tol * max(abs(x), abs(y))

economic_data = gcd_KPI_adequacy.pivot(
    index='day',
    columns = 'model_type',
    values=['OV_uc','OPEX_uc', 'LGEN_uc_MWh','EOV', 'E_OPEX', 'E_LOL_cost', 'E_LGEN_cost', 'slack_energy_reserve_up_cost_uc','slack_energy_reserve_down_cost_uc'],
    # var_name='cost_type',
    # value_name='cost_value'
)
# economic_data = economic_data.loc[filter_]

field_name = 'e_reserve_higher_than_envelope'
economic_data.loc[:,(field_name,'conservative')] = economic_data.apply(
    lambda row: is_bigger_than(row['OV_uc', 'e-reserve'], row['OV_uc', 'envelope']), axis=1)
economic_data.loc[:,(field_name,'envelope')]  = economic_data.loc[:,(field_name,'conservative')] 
economic_data.loc[:,(field_name,'e-reserve')]  = economic_data.loc[:,(field_name,'conservative')] 

field_name_2 = 'env_higher_than_cons'
economic_data.loc[:,(field_name_2,'conservative')] = economic_data.apply(
    lambda row: is_bigger_than(row['OV_uc', 'e-reserve'], row['OV_uc', 'conservative']), axis=1)
economic_data.loc[:,(field_name_2,'envelope')]  = economic_data.loc[:,(field_name_2,'conservative')] 
economic_data.loc[:,(field_name_2,'e-reserve')]  = economic_data.loc[:,(field_name_2,'conservative')]


field_name_3 = 'e_reserve_higher_than_conservative_RT'
economic_data.loc[:,(field_name_3,'conservative')] = economic_data.apply(
    lambda row: is_bigger_than(row['EOV', 'e-reserve'], row['EOV', 'conservative']), axis=1)
economic_data.loc[:,(field_name_3,'envelope')]  = economic_data.loc[:,(field_name_3,'conservative')] 
economic_data.loc[:,(field_name_3,'e-reserve')]  = economic_data.loc[:,(field_name_3,'conservative')]

economic_data = economic_data.stack('model_type', future_stack = True).reset_index()
print(economic_data[field_name].any())
print(economic_data[field_name_2].any())

In [ ]:
uc_columns = [col for col in gcd_KPI_adequacy.columns if '_uc' in col]
rest_columns = [col for col in gcd_KPI_adequacy.columns if '_uc' not in col]
# print("Columns containing '_uc':", uc_columns)
# print("Columns not containing '_uc':", rest_columns)
print("\nUC Columns ({}):".format(len(uc_columns)))
for col in uc_columns:
    print("  -", col)

print("\nOther Columns ({}):".format(len(rest_columns)))
for col in rest_columns:
    print("  -", col)

# Economic data

In [ ]:
fig = px.scatter(
    economic_data.melt(
        id_vars=['day', 'model_type'],
        value_vars=['OV_uc', 'EOV'],
        var_name='cost_type',
        value_name='cost'),
    x='day',
    y='cost',
    color='model_type',
    facet_row='cost_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
    # points="all",  # show all points
    # boxmode="group",
    # barmode = 'group',
    facet_col_spacing=0.07,
)


# Update facet layout spacing
fig.update_layout(
    # grid_xgap=0.1,  
    # autosize=True,
    width=1200,
    height=600,
    
)
fig.update_yaxes(matches=None)
# fig.for_each_xaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()

In [ ]:
fig = px.box(
    economic_data.melt(
        id_vars=['day', 'model_type'],
        value_vars=['OV_uc', 'OPEX_uc', 'EOV', 'E_OPEX', 'E_LOL_cost'],
        var_name='cost_type',
        value_name='cost'),
    x='model_type',
    y='cost',
    color='model_type',
    facet_col='cost_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
    # points="all",  # show all points
    boxmode="group",
    facet_col_spacing=0.07,
    hover_data='day'
)
fig.update_traces(
     boxmean=True, showlegend=False,
)

# Update facet layout spacing
fig.update_layout(
    # grid_xgap=0.1,  
    autosize=False,
    width=dim[0],
    # height=800,
)
fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()
# [fig.show(),fig.show()]


In [ ]:
to_plot = gcd_KPI_adequacy.groupby('model_type')[['start_cost_uc', 'fixed_cost_uc', 'production_cost_uc', 'storage_production_cost_uc', 'LGEN_cost_uc', 'LOL_cost_uc']].sum().round(2)
fig = px.bar(
    to_plot,
    barmode='stack',
    labels={'value': 'Cost ($)', 'variable': 'Cost Component', 'model_type': 'Model Type'},
    title='Cost Components by Model Type'
)
# update_background(fig, legend_attr, dim)
fig.show()

In [ ]:
#  to_plot.groupby(['model_type', 'value'])['cost'].agg(['mean', 'std']).reset_index()

In [ ]:
n = 10
fig.update_layout(margin=dict(l=n, r=n+20, t=n+10, b=n))
fig.write_image("day_ahead_cost_&_unserved_energy.pdf", width=dim[0], height=dim[1])
# fig.write_image("day_ahead_cost_&_unserved_energy.pdf", width=dim[0], height=dim[1])

In [ ]:
aux = gcd_KPI_adequacy[['model_type','OV_uc','E_ENS_MWh']].groupby('model_type').describe()
print((aux.loc['conservative',('OV_uc','mean')] - aux.loc['e-reserve',('OV_uc','mean')]) / aux.loc['conservative',('OV_uc','mean')] *100)
print((aux.loc['conservative',('OV_uc','mean')] - aux.loc['envelope',('OV_uc','mean')]) / aux.loc['conservative',('OV_uc','mean')] *100)
print((aux.loc['conservative',('E_ENS_MWh','mean')] - aux.loc['e-reserve',('E_ENS_MWh','mean')]) / aux.loc['e-reserve',('E_ENS_MWh','mean')] *100)
print((aux.loc['envelope',('E_ENS_MWh','mean')] - aux.loc['e-reserve',('E_ENS_MWh','mean')]) / aux.loc['e-reserve',('E_ENS_MWh','mean')] *100)


# Unserved energy and curtailment

In [ ]:
fig = px.scatter(
     economic_data.melt(
        id_vars=['day', 'model_type','EOV'],
        value_vars=['E_LOL_cost', 'E_LGEN_cost'],
        var_name='cost_type',
        value_name='cost'),
    y='cost',
    x='EOV',
    color='model_type',
    hover_data=['day'],
    facet_col= 'cost_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
)
fig.show()

In [ ]:
fig = px.box(
     gcd_KPI_adequacy.melt(
        id_vars=['day', 'model_type',],
        value_vars=['LGEN_uc_MWh', 'E_ENS_MWh', 'E_ENS_%', 'E_LGEN_MWh'],
        var_name='energy_type',
        value_name='energy'),
    x='model_type',
    y='energy',
    color='model_type',
    hover_data=['day'],
    facet_col= 'energy_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    # points="all",  # show all points
    boxmode="group",
    # facet_col_spacing=0.07,
    
)
fig.update_traces(boxmean=True, showlegend=False)
fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
# fig.show()
fig.show()

In [ ]:
aux = gcd_KPI_adequacy[['model_type','E_ENS_MWh','E_LLD_h','E_ENS_%']].groupby('model_type').describe()
print(aux)
print(gcd_KPI_adequacy[['model_type','E_ENS_MWh','E_LLD_h']].groupby('model_type').quantile([0.025, 1-0.025]))
# print((aux.loc['e-reserve',('E_ENS_MWh','mean')] - aux.loc['conservative',('E_ENS_MWh','mean')]) / aux.loc['conservative',('E_ENS_MWh','mean')] *100)
print(aux.loc['e-reserve',('E_ENS_%','mean')] - aux.loc['conservative',('E_ENS_%','mean')])


# Dispatch

In [ ]:
fig = px.box(
    dispatch_data.melt(
        id_vars=['day','model_type'],
        value_vars=['thermal_production_uc_MWh', 'storage_charge_uc_MWh', 'storage_discharge_uc_MWh'],
        var_name='dispatch_type',
        value_name='dispatch_value'
    ),
    x = 'model_type',
    color='model_type',
    y = 'dispatch_value',
    facet_col = 'dispatch_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
    # boxmode="group",
    # title="Storage charge+discharge UC distribution"
     facet_col_spacing=0.07,
)
fig.update_traces(
    boxmean=True,   showlegend=False,
    )  # show mean and standard deviation
fig.update_layout(
    # boxmean=True,
    legend = legend_attr,
    width=dim[0],
    height=dim[1],
)
fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.show()

# Reserves

In [ ]:
px.scatter(
    gcd_KPI_adequacy,
    x='day',
    y=['slack_reserve_up_cost_uc', 'slack_reserve_down_cost_uc', 'slack_energy_reserve_up_cost_uc', 'slack_reserve_down_cost_uc'],
    color='model_type',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
)


# Update facet layout spacing
# fig.update_layout(
#     # grid_xgap=0.1,  
#     # autosize=True,
#     width=1200,
#     height=600,
    
# )
# fig.update_yaxes(matches=None)
# fig.for_each_xaxis(lambda yaxis: yaxis.update(showticklabels=True))
# fig.show()

In [ ]:
def add_up_down_multiindex(df):
    new_columns = []
    for col in df.columns:
        col_str = str(col)
        if 'up' in col_str:
            # Remove 'up_' or '_up' from the column name
            if 'up_' in col_str:
                base_col = col_str.replace('up_', '')
            else:
                base_col = col_str
            new_columns.append(('up', base_col))
        elif 'down' in col_str:
            # Remove 'down_' or '_down' from the column name
            if 'down_' in col_str:
                base_col = col_str.replace('down_', '')
            else:
                base_col = col_str
            new_columns.append(('down', base_col))
        else:
            new_columns.append(('', col))
    df.columns = pd.MultiIndex.from_tuples(new_columns)
    df = df.stack(level=0, future_stack=True)
    df.index = df.index.set_names('reserve_direction', level=-1)
    return df


filtered_data = gcd_KPI_adequacy.pivot(
    index=['day'],
    columns = 'model_type',
    values=['thermal_reserve_up_uc_MWh', 'thermal_reserve_down_uc_MWh', 'storage_reserve_up_uc_MWh', 'storage_reserve_down_uc_MWh',
            'thermal_energy_reserve_up_uc_MWh','thermal_energy_reserve_down_uc_MWh', 'storage_energy_reserve_up_uc_MWh', 'storage_energy_reserve_down_uc_MWh',
            'required_reserve_up_uc_MWh', 'required_reserve_down_uc_MWh','required_energy_reserve_up_uc_MWh', 'required_energy_reserve_down_uc_MWh',
            'E_storage_reserve_up_activation_MWh', 'E_storage_reserve_down_activation_MWh',
            'E_thermal_reserve_up_activation_MWh', 'E_thermal_reserve_down_activation_MWh'],
    # var_name='cost_type',
    # value_name='cost_value'
)
filtered_data = filtered_data.loc[filter_]
# filtered_data = filtered_data.dropna()
# filtered_data.loc[:,('env_higher_than_cons','conservative')] = filtered_data['EOV','envelope'] > filtered_data['EOV','conservative']
# filtered_data.loc[:,('env_higher_than_cons','envelope')] = filtered_data['EOV','envelope'] > filtered_data['EOV','conservative']
# filtered_data.loc[:,('env_higher_than_cons','e-reserve')] = filtered_data['EOV','envelope'] > filtered_data['EOV','conservative']

filtered_data = filtered_data.stack('model_type', future_stack = True).reset_index()
filtered_data.set_index(['day', 'model_type'], inplace=True)

filtered_data = add_up_down_multiindex(filtered_data)




In [ ]:
filtered_data['thermal_reserve_commitment'] = (
    filtered_data['thermal_reserve_uc_MWh'].fillna(0) + filtered_data['thermal_energy_reserve_uc_MWh'].fillna(0)
) / (
    filtered_data['required_reserve_uc_MWh'].fillna(0) +
    # filtered_data['storage_reserve_uc_MWh'].fillna(0) +
    filtered_data['required_energy_reserve_uc_MWh'].fillna(0) 
    # filtered_data['storage_energy_reserve_uc_MWh'].fillna(0)
)
filtered_data['storage_reserve_commitment'] = (
    filtered_data['storage_reserve_uc_MWh'].fillna(0) + filtered_data['storage_energy_reserve_uc_MWh'].fillna(0)
) / (
    filtered_data['required_reserve_uc_MWh'].fillna(0) +
    # filtered_data['storage_reserve_uc_MWh'].fillna(0) +
    filtered_data['required_energy_reserve_uc_MWh'].fillna(0) 
    # filtered_data['storage_energy_reserve_uc_MWh'].fillna(0)
)

# This KPI is the ratio of the energy activation to the total required reserve and energy reserve. Good for checking if the reserve is being activated as expected and is lower than 1. Not useful for energy reserves.
filtered_data['E_thermal_reserve_activation'] = filtered_data['E_thermal_reserve_activation_MWh'].fillna(0) / (filtered_data['required_reserve_uc_MWh'].fillna(0) + filtered_data['required_energy_reserve_uc_MWh'].fillna(0))

filtered_data['E_storage_reserve_activation'] = filtered_data['E_storage_reserve_activation_MWh'].fillna(0) / (filtered_data['required_reserve_uc_MWh'].fillna(0) + filtered_data['required_energy_reserve_uc_MWh'].fillna(0))


In [ ]:
legend_attr = dict(
    x=0.45,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


# Create the stacked bar plot with bar style
fig = px.bar(
    filtered_data.reset_index(),
    x='day',
    y=['storage_reserve_commitment'], # 'storage_reserve_commitment'
    color='model_type',
    barmode='group',
    facet_row='reserve_direction',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis2=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title=''),
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title=''),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="day"),
    width=dim[0],
    height=dim[1],
    title_text="Storage Reserve Commitment Ratio",
    # showlegend=True,
    legend=legend_attr  # Include legend attributes
)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

fig.show()



In [ ]:
# fig = px.box(
#     filtered_data.reset_index().melt(
#         id_vars=['model_type','reserve_direction'],
#         value_vars=['storage_reserve_commitment', 'thermal_reserve_commitment']),
#     facet_col='variable',
#     y='value',
#     color='model_type',
#     x='reserve_direction',
#     category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
#     points="all",  # show all points
#     boxmode="group",
# )

# fig.update_traces(
#     boxmean=True,
#     # boxmean='sd'
#     )  # show mean and standard deviation

# fig.show()

In [ ]:
filtered_data.reset_index().melt(
    id_vars=['day','model_type','reserve_direction'],
    value_vars=['storage_reserve_commitment', 'thermal_reserve_commitment','E_thermal_reserve_activation','E_storage_reserve_activation']
)

In [ ]:
fig = px.box(
    filtered_data.reset_index().melt(
    id_vars=['day','model_type','reserve_direction'],
    value_vars=['storage_reserve_commitment', 'thermal_reserve_commitment','E_thermal_reserve_activation','E_storage_reserve_activation']),
    x='reserve_direction',
    y='value',
    color='model_type',
    facet_col='variable',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
    # points="all",  # show all points
    boxmode="group",
)
fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))
fig.update_traces(
    boxmean=True,
    # boxmean='sd'
    )  # show mean and standard deviation

fig.show()

In [ ]:
scale = .7
dim = (1000*scale,500*scale)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)
fig = px.box(
    filtered_data.reset_index(),
    # x='model_type',
    y='storage_reserve_commitment',
    color='model_type',
    x='reserve_direction',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},
    # points="all",  # show all points
    boxmode="group",
)
fig.update_traces(
     boxmean=True, showlegend=True,
)
fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title = 'Storage reserve commitment'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title = ''),
    width=dim[0],
    height=dim[1],
    # showlegend=True,
    autosize=False,
    legend=legend_attr  # Include legend attributes
)
# Add outline boxes to each subplot
# for axis in fig.layout:
#     if axis.startswith('xaxis') or axis.startswith('yaxis'):
#         fig.layout[axis].update(showline=True, linecolor="grey", mirror=True, title = '')
# Update facet layout spacing

fig.update_yaxes(matches=None)
fig.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True))

fig.show()

In [ ]:
n = 10
fig.update_layout(margin=dict(l=n, r=n, t=n, b=n))
fig.write_image("storage_reserve_participation.pdf", width=dim[0], height=dim[1])

In [ ]:
legend_attr = dict(
    x=0.45,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig = px.bar(
    filtered_data.reset_index(),
    x='day',
    y=['E_storage_reserve_activation_MWh'], # 'storage_reserve_commitment'
    color='model_type',
    barmode='group',
    facet_row='reserve_direction',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
)
fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis2=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='[MWh/day]'),
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='[MWh/day]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="day"),
    width=dim[0],
    height=dim[1],
    title_text="Storage Reserve RT Activation",
    # showlegend=True,
    legend=legend_attr  # Include legend attributes
)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

fig.show()

In [ ]:
legend_attr = dict(
    x=0.45,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

fig = px.bar(
    filtered_data.reset_index(),
    x='day',
    y=['E_thermal_reserve_activation_MWh'], # 'storage_reserve_commitment'
    color='model_type',
    barmode='group',
    facet_row='reserve_direction',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis2=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="[MWh/day]"),
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title= "[MWh/day]"),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="day"),
    width=dim[0],
    height=dim[1],
    title_text="Thermal Reserve RT Activation",
    # showlegend=True,
    legend=legend_attr  # Include legend attributes
)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

fig.show()


In [ ]:
# # Create the stacked bar plot with bar style
# fig = px.bar(
#     filtered_data,
#     x='day',
#     y=['E_storage_reserve_down_activation_MWh'],
#     color='model_type',
#     # hover_data=['day'],
#     # color_discrete_sequence=[f'#{g_BLUE}', f'#{g_GREY}'],
#     barmode='group',
#     # facet_col='env_higher_than_cons',
#     category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
#     # labels={'Cost_Type': 'cost', 'model_type': 'model'},
#     # pattern_shape='Cost_Type',
#     pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
# )
# fig.show()

In [ ]:
legend_attr = dict(
    x=0.45,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

# Create the stacked bar plot with bar style
fig = px.bar(
    economic_data,
    x='day',
    y='EOV',
    color='model_type',
    barmode='group',
    facet_col=field_name_3,
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order

)

fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='RT system cost [$]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="day"),
    width=dim[0],
    height=dim[1],
    # showlegend=True,
    legend=legend_attr  # Include legend attributes
)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)

fig.show()

In [ ]:
filtered_data 

In [ ]:
legend_attr = dict(
    x=0.45,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

filtered_data = gcd_KPI_adequacy.melt(
    id_vars=['model_type', 'day'],
    value_vars=['E_OPEX', 'E_LOL_cost', 'E_LGEN_cost'],
    var_name='Cost_Type',
    value_name='Cost_Value'
)

days_to_plot = range(1,11)
# Create the stacked bar plot with bar style
fig = px.bar(
    filtered_data[filtered_data.day.isin(days_to_plot)],
    x='model_type',
    y='Cost_Value',
    color='model_type',
    barmode='stack',
    facet_col='day',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    labels={'Cost_Type': 'cost', 'model_type': 'model'},
    pattern_shape='Cost_Type',
    pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
)


# Update layout for better visualization
fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='RT system cost [$]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="ρ"),
    width=dim[0]*2,
    height=dim[1],
    # showlegend=True,
    # legend=legend_attr  # Include legend attributes
)
# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)
# Show the plot
fig.update_xaxes(showticklabels=False, title=None)
# fig.update_yaxes(range=[790000, 822000])
fig.show()


In [ ]:
if G_save:
    fig.write_image("total_system_costs.pdf", width=dim[0], height=dim[1])

In [ ]:
filtered_data

In [ ]:
legend_attr = dict(
    x=0.45,
    y=-3,
    yanchor="bottom",
    xanchor="center",
    orientation="v"
)

filtered_data = gcd_KPI_adequacy.melt(
    id_vars=['model_type', 'day'],
    value_vars=['OPEX_uc', 'slack_reserve_up_cost_uc', 'slack_reserve_down_cost_uc', 'slack_energy_reserve_up_cost_uc', 'slack_energy_reserve_down_cost_uc'],
    var_name='Cost_Type',
    value_name='Cost_Value'
)
filtered_data.fillna(0, inplace=True)  # Fill NaN values with 0 for better visualization
# Create the stacked bar plot with bar style
fig = px.bar(
    filtered_data.query("day in @days_to_plot"),
    x='model_type',
    y='Cost_Value',
    color='model_type',
    barmode='stack',
    facet_col='day',
    category_orders={"model_type": ["conservative", "envelope", "e-reserve", "stochastic"]},  # Define the order
    labels={'Cost_Type': 'cost', 'model_type': 'model'},
    pattern_shape='Cost_Type',
    pattern_shape_sequence=['', '/', '\\', 'x', '-', '|', '+', '.'],
)


# Update layout for better visualization
fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title='DA system cost [$]'),
    xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True, title="ρ"),
    width=dim[0]*2,
    height=dim[1],
    # showlegend=True,
    # legend=legend_attr  # Include legend attributes
)
# Add outline boxes to each subplot
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(showline=True, linecolor="grey", mirror=True)
# Show the plot
fig.update_xaxes(showticklabels=False, title=None)
# fig.update_yaxes(range=[790000, 822000])
fig.show()


In [ ]:
model = 'uc'
days = range(1,365)
scalar = []
scalar_ =pd.DataFrame()
for sol in ss:
    s = sol['solution_folder']
    for day in days:
        try:
            scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{day}',f's_{model}_scalar.parquet'))
            print(f'(ss, days):{s}, n_{day}')
            scalar_['day'] = day
            scalar_['solution_id'] = sol['solution_folder']
            scalar.append(scalar_)
        except Exception:
            pass
    
    days_str = "-".join(str(d) for d in days)
    try:
        scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{days_str}', f's_{model}_scalar.parquet'))
        print(f'(ss, days):{s}, n_{days_str}')
        # scalar_['day'] = 0
        scalar_['solution_id'] = sol['solution_folder']
        scalar.append(scalar_)
    except Exception:
        pass

scalar = pd.concat(scalar)  

In [ ]:
scalar.loc[scalar['relative_gap_discrete_model'] >=1e-8,:].describe()

In [ ]:
gcd_KPI_adequacy = gcd_KPI_adequacy.merge(
    scalar[['solution_id', 'configuration', 'day', 'objective_value', 'relative_gap_discrete_model']],
    on=['solution_id', 'configuration', 'day'],
    suffixes=('', '_s')
)

In [ ]:
gcd_KPI_adequacy

In [ ]:
px.scatter(gcd_KPI_adequacy, x='E_LOL_cost', y='E_LGEN_cost', color='relative_gap_discrete_model')